# Notebook 0: FERP Data Acquisition

This is the first notebook in the sequence for the HRS Botany project.

In it we will download and clean the tree census data from the UCSC Forest Ecology Research Plot. 

Please reference the _notebooks/ferp_eda.ipynb_ notebook for FERP EDA.

__Notebook Inputs:__
- FERP Tree Data (_FERP123merged_20231029.csv_)
- FERP Tree Species Schema (_ferp_tree_species.txt_)

Notebook Outputs:
- FERP Cleaned Data (_data/ferp_trees.csv_)


In [9]:
# Imports

import numpy as np
import pandas as pd

import geopandas as gpd
from shapely.geometry import Point

from hrs_botany.data_utils import load_ferp_species_table

### Import the FERP trees dataset.

In [2]:
ferp_path = '../../data/ferp/geoforest/doi_10_5061_dryad_6q573n64s__v20240129/FERP123merged_20231029.csv'

# Load the dataset
df = pd.read_csv(ferp_path)

# Overview
print(df.shape)
print(df.columns)

(51016, 34)
Index(['quadrat', 'tag', 'stemtag', 'stemtag1', 'code6', 'east_m', 'north_m',
       'east_UTM', 'north_UTM', 'dsh1_mm', 'dsh2_mm', 'dsh3_mm', 'dsh1m_mm',
       'date1', 'date2', 'date3', 'status1', 'condition1', 'status2',
       'condition2', 'status3', 'condition3', 'first_census', 'irreg_dsh',
       'hom_m', 'multi1', 'stems1', 'multi2', 'multi3', 'basalarea1_m2',
       'code6fix', 'locfix', 'notes2', 'notes3'],
      dtype='object')


/var/folders/qx/bpj16cl90cq20swjd4j79cdh0000gn/T/ipykernel_45069/1956600545.py:4: DtypeWarning: Columns (13,16,17,23,25,27,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ferp_path)


### Now import the FERP Tree species mappings

In [6]:
ferp_species_path = '../../data/ferp/ferp_tree_species.txt'

df_species = load_ferp_species_table(ferp_species_path)

df_species.head(3)

,Scientific name,Common name,Code,Family,Related,Genus,Specific epithet,Author
0,Acer macrophyllum Pursh,Big-leaf maple,ACERMA,Sapindaceae,,Acer,macrophyllum,Pursh
1,Adenostoma fasciculatum Hook. & Arn.,Chamise,ADENFA,Rosaceae,,Adenostoma,fasciculatum,Hook. & Arn.
2,Arbutus menziesii Pursh,Madrone,ARBUME,Ericaceae,,Arbutus,menziesii,Pursh


### Initial data cleaning

In [7]:
common_name_d = df_species.set_index('Code')['Common name'].to_dict()
specific_epithet_d = df_species.set_index('Code')['Specific epithet'].to_dict()
genus_d = df_species.set_index('Code')['Genus'].to_dict()

df['common_name'] = df['code6'].map(common_name_d)
df['scientific_name'] = df['code6'].map(genus_d) + ' ' + df['code6'].map(specific_epithet_d)

In [11]:
# Manual list of tree species to filter

tree_species = [
    "Shreve’s oak", "Douglas-fir", "Coast redwood", "Tanoak",
    "California hazelnut", "Coast live oak", "Madrone", "California Bay",
    "Ponderosa pine", "Knobcone pine", "Big-leaf maple", "Yellow willow",
    "Loquat", "Blue-gum eucalyptus"
]

df = df[df['common_name'].isin(tree_species)]

len(df)

41229

In [12]:
df.to_csv('data/ferp_trees.csv', index=False)

In [13]:
df

,quadrat,tag,stemtag,stemtag1,code6,east_m,north_m,east_UTM,north_UTM,dsh1_mm,...,stems1,multi2,multi3,basalarea1_m2,code6fix,locfix,notes2,notes3,common_name,scientific_name
0,E000_N000,2,1.0,NaN,QUERPA,2.6,6.7,582309.51,4096655.62,31.0,...,1.0,NaN,NaN,0.000755,NaN,loc_fixed,NaN,Tag 2 was not on the original data sheet even ...,Shreve’s oak,Quercus parvula
1,E000_N000,3,1.0,NaN,PSEUME,0.6,6.2,582307.45,4096655.65,378.0,...,1.0,NaN,NaN,0.112221,NaN,NaN,NaN,NaN,Douglas-fir,Pseudotsuga menziesii
2,E000_N000,4,1.0,NaN,QUERPA,0.5,6.8,582307.51,4096656.25,20.0,...,1.0,NaN,NaN,0.000314,NaN,NaN,NaN,NaN,Shreve’s oak,Quercus parvula
3,E000_N000,5,1.0,NaN,SEQUSE,3.1,13.7,582311.77,4096662.27,1420.0,...,1.0,NaN,NaN,1.583677,NaN,NaN,NaN,NaN,Coast redwood,Sequoia sempervirens
4,E000_N000,6,1.0,NaN,QUERPA,4.7,19.2,582314.71,4096667.18,74.0,...,1.0,multi2,NaN,0.004301,NaN,NaN,NaN,NaN,Shreve’s oak,Quercus parvula
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51011,E280_N320,34371,1.0,NaN,PSEUME,298.7,327.6,582677.22,4096891.09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Douglas-fir,Pseudotsuga menziesii
51012,E280_N300,34694,1.0,NaN,QUERPA,299.4,303.4,582671.76,4096867.51,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Shreve’s oak,Quercus parvula
51013,E320_N360,35287,1.0,NaN,SEQUSE,339.6,374.8,582728.73,4096926.40,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Coast redwood,Sequoia sempervirens
51014,E340_N040,36362,1.0,NaN,LITHDE,346.7,42.6,582651.49,4096603.23,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Tanoak,Notholithocarpus densiflorus


In [14]:
df.columns

Index(['quadrat', 'tag', 'stemtag', 'stemtag1', 'code6', 'east_m', 'north_m',
       'east_UTM', 'north_UTM', 'dsh1_mm', 'dsh2_mm', 'dsh3_mm', 'dsh1m_mm',
       'date1', 'date2', 'date3', 'status1', 'condition1', 'status2',
       'condition2', 'status3', 'condition3', 'first_census', 'irreg_dsh',
       'hom_m', 'multi1', 'stems1', 'multi2', 'multi3', 'basalarea1_m2',
       'code6fix', 'locfix', 'notes2', 'notes3', 'common_name',
       'scientific_name'],
      dtype='object')